# 歌词分词，词性标注

In [1]:
import json
import pandas as pd


# import thulac


from collections import Counter
from openai import OpenAI

In [20]:
# 可以选择是否加载
# jieba.load_userdict('data/mayday_dict_simple.txt')

In [60]:
import sys, os
sys.path.append('..')

# 分词，词频与词性分析

In [3]:
word_to_fix = {
    '阮': 'r',
    '袂': 'v',
    '春娇': 'n',
    '学会': 'v'
}

In [4]:
# def process_lyrics_with_jieba(text):
#     # 1. 词性标注与分词
#     # jieba.posseg 会同时返回词和词性
#     words_with_pos = pseg.cut(text)

    
#     # 2. 过滤无意义字符（标点、空格、单字符停用词）
#     filtered_data = []
#     for word, pos in words_with_pos:
#         # 排除标点符号（x表示标点）及空白字符
#         if pos != 'x' and len(word.strip()) > 0:
#             if word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 3. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 4. 汇总信息 (词, 词性, 频数)
#     # 我们以词为 Key，存储词性
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     # 排序：按词频从高到低
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count # 词频
#         })
    
#     return sorted_results

In [5]:
import re
from hanlp_restful import HanLPClient


api = "https://hanlp.com/hanlp/v21/redirect"
# api = "https://hanlp.hankcs.com/api"
# api = "https://www.hanlp.com/api"
HanLP = HanLPClient(api, auth="699691e7eaf61a3aca90d7b8", language='zh')


def is_chinese_word(word):
    """
    判断是否为纯中文词
    """
    return 1 if re.fullmatch(r'[\u4e00-\u9fff]+', word) else 0

def is_english_word(word):
    """
    判断是否为纯英文词
    """
    return 1 if re.fullmatch(r'[a-zA-Z]+', word) else 0



def process_lyrics_with_hanlp_multi_pos(text, word_to_fix=None):
    if not text:
        return []

    # 调用 HanLP
    result = HanLP.parse(text, tasks='pos/pku')

    sentences = result['tok/fine']
    pos_sentences = result['pos/pku']

    # 统计 (word, pos) -> freq
    word_pos_counter = Counter()

    for words, pos_tags in zip(sentences, pos_sentences):
        for word, tag in zip(words, pos_tags):

            word = word.strip()

            # 过滤标点
            if tag == 'w' or not word:
                continue

            # 词性修正
            if word_to_fix and word in word_to_fix:
                tag = word_to_fix[word]

            word_pos_counter[(word, tag)] += 1

    # 构建结果列表
    results = []
    for (word, pos), freq in word_pos_counter.items():
        results.append({
            "word": word,
            "pos": pos,
            "freq": freq,
            "is_chinese": is_chinese_word(word)
        })

    # 按词频排序
    results.sort(key=lambda x: x["freq"], reverse=True)

    return results


In [6]:
# thu = thulac.thulac(seg_only=False, filt=True) 

# def process_lyrics_with_thulac(text, word_to_fix=None):
#     if not text:
#         return []
    
#     # 2. 执行分词与词性标注
#     # 返回格式为 [[word, pos], [word, pos], ...]
#     words_with_pos = thu.cut(text)
    
#     # 3. 过滤无意义字符与词性修正
#     # thulac 的标点词性通常是 'w'
#     filtered_data = []
#     for word, pos in words_with_pos:
#         word = word.strip()
#         # 排除标点符号、空白字符
#         if pos != 'w' and len(word) > 0:
#             # 逻辑修正：word_to_fix 通常是修正词性
#             if word_to_fix and word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 4. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 5. 汇总信息
#     # 建立 word -> pos 映射
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count
#         })
    
#     return sorted_results

In [64]:
def lyric_words_process(path_prefix, word_to_fix=None):
    lyric_file_path = path_prefix + 'cleared_lyric_data.json'

    # 已解析的内容
    songs_id_exist = []
    if os.path.exists(path_prefix + "raw_words_data.csv"):
        df_exist = pd.read_csv(path_prefix + "raw_words_data.csv")
        songs_id_exist = df_exist['song_id'].unique().tolist()

    # 读取歌词文件
    with open(lyric_file_path, 'r') as f:
        lyric_data = json.load(f)
    lyric_words_dict = {}
    for i in lyric_data:
        if i and i['song_id'] not in songs_id_exist:
            # lyric_words_dict[i['song_id']] = process_lyrics_with_jieba(
            #     i['lyrics_text'])
            # lyric_words_dict[i['song_id']] = process_lyrics_with_thulac(
            #     i['lyrics_text'], word_to_fix=word_to_fix)
            print(i['song_name'])
            lyric_words_dict[i['song_id']] = process_lyrics_with_hanlp_multi_pos(
                i['lyrics_text'], word_to_fix=word_to_fix)
    rows = []
    for song_id, word_list in lyric_words_dict.items():
        for item in word_list:
            # 创建新字典，保留原始数据并加入歌曲ID列
            new_row = {
                'song_id': song_id,
                'word': item['word'],
                'pos': item['pos'],
                'freq': item['freq']
            }
            rows.append(new_row)

    # 3. 转换为 DataFrame
    df_word = pd.DataFrame(rows)
    # 合并df_exis和 df_word
    if 'df_exist' in locals():
        df_word = pd.concat([df_exist, df_word], ignore_index=True)
    return df_word

In [8]:
def words_data_merge(df_word, df_songs):
    # 合并
    # 1. 确保 df_word 的 song_id 是字符串
    df_word = df_word.copy()
    df_songs = df_songs.copy()
    df_word['song_id'] = df_word['song_id'].astype(str)
    df_word['word'] = df_word['word'].astype(str)
    df_word['is_chinese'] = df_word['word'].apply(is_chinese_word)
    df_word['is_english'] = df_word['word'].apply(is_english_word)
    # 把英文词转为小写
    df_word.loc[df_word['is_english'] == 1, 'word'] = df_word.loc[df_word['is_english'] == 1, 'word'].str.lower()
    # 2. 确保 df_unique 的 song_id 是字符串（并去掉可能存在的空格）
    df_songs['song_id'] = df_songs['song_id'].astype(str).str.strip()

    # 3. 执行合并
    df_merged = df_word.merge(df_songs, on='song_id', how='left')

    # 4. 删除空值
    # df_merged = df_merged.dropna()

    return df_merged

# 批量采集

In [ ]:
singers = [('luodayou', '罗大佑'), ('lizongsheng', '李宗盛'), ('zhangxueyou', '张学友'), ('twins', 'Twins'), ('wangsulong', '汪苏泷'), ('panweibo', '潘玮柏'), ('dengziqi', 'G.E.M. 邓紫棋'), ('xuezhiqian', '薛之谦'), ('xusong', '许嵩'), ('zhangjie', '张杰'), ('taozhe', '陶喆'), ('fangdatong', '方大同'), ('wangfei', '王菲'), ('maobuyi', '毛不易'), ('beyond', 'BEYOND')]
for i in singers[-1:]:
    file_path_prefix = f'data/{i[0]}/'
    # 歌曲数据
    df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
    # 词性解析
    # hanlp暂时不需要word_to_fix
    df_word = lyric_words_process(file_path_prefix, word_to_fix=None)
    df_word.to_csv(file_path_prefix + "raw_words_data.csv", index=False)
    # 重新读取
    df_word_read = pd.read_csv(file_path_prefix + "raw_words_data.csv")
    df_merged = words_data_merge(df_word_read, df_songs)
    df_merged = df_merged.dropna(subset='song_name')
    # 过滤中文词
    df_merged_chn = df_merged[df_merged['is_chinese'] == 1]
    # 虚拟专辑数据
    df_songs_part = df_merged_chn[[
        'song_name_pure'
    ]].drop_duplicates(keep='first').reset_index(drop=True)
    # 只保留120个
    df_songs_part = df_songs_part.head(100)
    df_songs_part['album_name'] = "PART " + (df_songs_part.index // 10 +
                                            1).astype(str)
    df_songs_part['album_order'] = df_songs_part.index // 10
    # 虚拟专辑数据，index//12+1作为虚拟专辑
    df_merged_chn = df_merged_chn.copy()
    df_merged_chn['album_name_raw'] = df_merged_chn['album_name']
    df_merged_chn = df_merged_chn.drop(columns=['album_name'])
    df_merged_chn = df_merged_chn.merge(df_songs_part, on='song_name_pure', how='left')
    # 删除album_order为空的数据
    df_merged_chn = df_merged_chn.dropna(subset=['album_order'], axis=0)
    df_merged_chn.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)
    df_songs_final = df_merged_chn.drop(columns=['word', 'pos', 'freq', 'is_chinese']).drop_duplicates().reset_index(drop=True)
    df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

# main

In [78]:
singer_list = [
        'mayday', 'jaychou', 'liyuchun', 'chenyixun', 'renxianqi', 'linjunjie',
        'sunyanzi', 'remen', 'fangwenshan', 'chenxinhong', 'caiyilin', 'wubai', 'zhoushen', 'zhoushen_pure', 'fenghuangchuanqi', 'wanglihong', 'liangjingru', 'wangxinling', 'twins', 'beyond', 'wuyuetian', 'luodayou', 'fangdatong', 'taozhe', 'lizongsheng', 'mowenwei', 'fangdatong', 'wangsulong', 'maobuyi', 'zhoujielun', 'suyoupeng'
    ]
file_path_prefix = f"data/{singer_list[-1]}/"
file_path_prefix = f"data/liangjingru/"

In [79]:
# 歌曲数据
df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year
0,462188,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,44,000GGDys0yA0Nk,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009
1,411228,004K6Ne61a1VA8,会呼吸的痛,NaN,梁静茹,44,000GGDys0yA0Nk,崇拜,33237,002oy2Mp3I8Rgo,272,1194537600,会呼吸的痛,会呼吸的痛,崇拜,2007-11-09,2007
2,411231,002x8dpU2QNXFP,给未来的自己,NaN,梁静茹,44,000GGDys0yA0Nk,崇拜,33237,002oy2Mp3I8Rgo,250,1194537600,给未来的自己,给未来的自己,崇拜,2007-11-09,2007
3,4830164,000LDr7E13dXy1,勇气,《侠女闯天关》电视剧台湾版主题曲|《出水芙蓉》电视剧片尾曲,梁静茹,44,000GGDys0yA0Nk,勇气,96253,001DEgPu1004bl,239,965145600,勇气,勇气,勇气,2000-08-02,2000
4,169763,003Sn9Rg4N3oNq,暖暖,《周末父母》电视剧片头曲,梁静茹,44,000GGDys0yA0Nk,亲亲,14922,004R8LFN08EGAD,243,1160064000,暖暖,暖暖,亲亲,2006-10-06,2006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,321887559,004dUmuD0LyeM6,明天，双人舞,NaN,梁静茹,44,000GGDys0yA0Nk,时光随想·三日思,21864867,003I0tPA3u1TNA,187,1630468800,明天双人舞,明天双人舞,时光随想·三日思,2021-09-01,2021
126,462193,0006yWkR3loMtl,找个人,NaN,梁静茹,44,000GGDys0yA0Nk,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,218,1232035200,找个人,找个人,静茹 & 情歌 别再为他流泪,2009-01-16,2009
127,108784387,001hRCHO49usEC,呵护,NaN,梁静茹,44,000GGDys0yA0Nk,呵护,1634993,003coBCr3ZwJFf,232,1475942400,呵护,呵护,呵护,2016-10-09,2016
128,4932196,004ZWC2P1sPDFT,Tiffany,NaN,梁静茹,44,000GGDys0yA0Nk,恋爱的力量,96351,002N40J82N3yaM,243,1069689600,Tiffany,tiffany,恋爱的力量,2003-11-25,2003


In [ ]:
# 五月天需要使用word_to_fix
# if file_path_prefix == "data/mayday/":
#     df_word = lyric_words_process(file_path_prefix, word_to_fix)
# else:
#     df_word = lyric_words_process(file_path_prefix, word_to_fix=None)

In [80]:
# 词性解析
# hanlp暂时不需要word_to_fix
df_word = lyric_words_process(file_path_prefix, word_to_fix=None)
df_word.to_csv(file_path_prefix + "raw_words_data.csv", index=False)
df_word

,song_id,word,pos,freq
0,462188,的,u,17
1,462188,我,r,15
2,462188,着,u,7
3,462188,一,m,6
4,462188,你,r,5
...,...,...,...,...
12731,797249,个人,n,1
12732,797249,地方,n,1
12733,797249,一切,r,1
12734,797249,都,d,1


In [81]:
# 重新读取
df_word_read = pd.read_csv(file_path_prefix + "raw_words_data.csv")
df_word_read

,song_id,word,pos,freq
0,462188,的,u,17
1,462188,我,r,15
2,462188,着,u,7
3,462188,一,m,6
4,462188,你,r,5
...,...,...,...,...
12731,797249,个人,n,1
12732,797249,地方,n,1
12733,797249,一切,r,1
12734,797249,都,d,1


In [82]:
df_merged = words_data_merge(df_word_read, df_songs)
df_merged = df_merged.dropna(subset='song_name')
df_merged

,song_id,word,pos,freq,is_chinese,is_english,song_mid,song_name,song_subname,artist_name,...,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year
0,462188,的,u,17,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009
1,462188,我,r,15,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009
2,462188,着,u,7,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009
3,462188,一,m,6,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009
4,462188,你,r,5,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12731,797249,个人,n,1,1,0,001gusT73lIgei,蔚蓝海岸,NaN,梁静茹,...,现在开始我爱你,69903,003LwedV3ZJrkg,290,1296489600,蔚蓝海岸,蔚蓝海岸,现在开始我爱你,2011-02-01,2011
12732,797249,地方,n,1,1,0,001gusT73lIgei,蔚蓝海岸,NaN,梁静茹,...,现在开始我爱你,69903,003LwedV3ZJrkg,290,1296489600,蔚蓝海岸,蔚蓝海岸,现在开始我爱你,2011-02-01,2011
12733,797249,一切,r,1,1,0,001gusT73lIgei,蔚蓝海岸,NaN,梁静茹,...,现在开始我爱你,69903,003LwedV3ZJrkg,290,1296489600,蔚蓝海岸,蔚蓝海岸,现在开始我爱你,2011-02-01,2011
12734,797249,都,d,1,1,0,001gusT73lIgei,蔚蓝海岸,NaN,梁静茹,...,现在开始我爱你,69903,003LwedV3ZJrkg,290,1296489600,蔚蓝海岸,蔚蓝海岸,现在开始我爱你,2011-02-01,2011


In [83]:
df_merged[df_merged['pos'] == 'e']['word'].unique()

array(['哦', 'hi', '呼', 'hey', '哼', '喔', '噢噢噢', '嘿', '哎', '噢', '哈', 'oh',
       '诶'], dtype=object)

In [84]:
# 过滤中文词
df_merged_chn = df_merged[df_merged['is_chinese'] == 1]
df_merged_chn
# 不过滤中文词
# df_merged_chn = df_merged.copy()
# 只保留中文和英文
# df_merged_chn = df_merged_chn[(df_merged_chn['is_chinese'] == 1) | (df_merged_chn['is_english'] == 1)]

,song_id,word,pos,freq,is_chinese,is_english,song_mid,song_name,song_subname,artist_name,...,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year
0,462188,的,u,17,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009
1,462188,我,r,15,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009
2,462188,着,u,7,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009
3,462188,一,m,6,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009
4,462188,你,r,5,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,静茹 & 情歌 别再为他流泪,37603,002E4IXe1ESUij,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12731,797249,个人,n,1,1,0,001gusT73lIgei,蔚蓝海岸,NaN,梁静茹,...,现在开始我爱你,69903,003LwedV3ZJrkg,290,1296489600,蔚蓝海岸,蔚蓝海岸,现在开始我爱你,2011-02-01,2011
12732,797249,地方,n,1,1,0,001gusT73lIgei,蔚蓝海岸,NaN,梁静茹,...,现在开始我爱你,69903,003LwedV3ZJrkg,290,1296489600,蔚蓝海岸,蔚蓝海岸,现在开始我爱你,2011-02-01,2011
12733,797249,一切,r,1,1,0,001gusT73lIgei,蔚蓝海岸,NaN,梁静茹,...,现在开始我爱你,69903,003LwedV3ZJrkg,290,1296489600,蔚蓝海岸,蔚蓝海岸,现在开始我爱你,2011-02-01,2011
12734,797249,都,d,1,1,0,001gusT73lIgei,蔚蓝海岸,NaN,梁静茹,...,现在开始我爱你,69903,003LwedV3ZJrkg,290,1296489600,蔚蓝海岸,蔚蓝海岸,现在开始我爱你,2011-02-01,2011


In [85]:
# 查看歌曲数
df_merged_chn['song_name_pure'].nunique()

130

In [86]:
# 虚拟专辑数据
df_songs_part = df_merged_chn[[
    'song_name_pure'
]].drop_duplicates(keep='first').reset_index(drop=True)
# 只保留120个
df_songs_part = df_songs_part.head(100)
df_songs_part['album_name'] = "PART " + (df_songs_part.index // 10 +
                                         1).astype(str)
df_songs_part['album_order'] = df_songs_part.index // 10
df_songs_part

,song_name_pure,album_name,album_order
0,情歌,PART 1,0
1,会呼吸的痛,PART 1,0
2,给未来的自己,PART 1,0
3,勇气,PART 1,0
4,暖暖,PART 1,0
...,...,...,...
95,看海计划,PART 10,9
96,不敢当,PART 10,9
97,麋鹿,PART 10,9
98,宁静海,PART 10,9


In [87]:
# 虚拟专辑数据，index//12+1作为虚拟专辑
df_merged_chn = df_merged_chn.copy()
df_merged_chn['album_name_raw'] = df_merged_chn['album_name']
df_merged_chn = df_merged_chn.drop(columns=['album_name'])
df_merged_chn = df_merged_chn.merge(df_songs_part, on='song_name_pure', how='left')
# 删除album_order为空的数据
df_merged_chn = df_merged_chn.dropna(subset=['album_order'], axis=0)
df_merged_chn

,song_id,word,pos,freq,is_chinese,is_english,song_mid,song_name,song_subname,artist_name,...,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year,album_name_raw,album_name,album_order
0,462188,的,u,17,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009,静茹 & 情歌 别再为他流泪,PART 1,0.0
1,462188,我,r,15,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009,静茹 & 情歌 别再为他流泪,PART 1,0.0
2,462188,着,u,7,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009,静茹 & 情歌 别再为他流泪,PART 1,0.0
3,462188,一,m,6,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009,静茹 & 情歌 别再为他流泪,PART 1,0.0
4,462188,你,r,5,1,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,...,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009,静茹 & 情歌 别再为他流泪,PART 1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9923,412406457,何止,v,1,1,0,001OBHfh3Z3yyo,来日不方长,NaN,梁静茹,...,215,1684339200,来日不方长,来日不方长,麋鹿,2023-05-18,2023,麋鹿,PART 10,9.0
9924,412406457,家乡,n,1,1,0,001OBHfh3Z3yyo,来日不方长,NaN,梁静茹,...,215,1684339200,来日不方长,来日不方长,麋鹿,2023-05-18,2023,麋鹿,PART 10,9.0
9925,412406457,蝉声,n,1,1,0,001OBHfh3Z3yyo,来日不方长,NaN,梁静茹,...,215,1684339200,来日不方长,来日不方长,麋鹿,2023-05-18,2023,麋鹿,PART 10,9.0
9926,412406457,弥漫,v,1,1,0,001OBHfh3Z3yyo,来日不方长,NaN,梁静茹,...,215,1684339200,来日不方长,来日不方长,麋鹿,2023-05-18,2023,麋鹿,PART 10,9.0


In [88]:
df_merged_chn.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)

In [89]:
# 数据查验
songs_n = df_merged_chn[df_merged_chn['pos'] == 'n']['song_name_pure'].unique().tolist()
songs_all = df_merged_chn['song_name_pure'].unique().tolist()
for i in songs_all:
    if i not in songs_n:
        print(i)

# 歌曲数据更新

In [90]:
df_songs_final = df_merged_chn.drop(columns=['word', 'pos', 'freq', 'is_chinese']).drop_duplicates().reset_index(drop=True)

df_songs_final

,song_id,is_english,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year,album_name_raw,album_name,album_order
0,462188,0,0010BrWk2SucQr,情歌,《败犬女王》电视剧插曲,梁静茹,44,000GGDys0yA0Nk,37603,002E4IXe1ESUij,260,1232035200,情歌,情歌,静茹 & 情歌 别再为他流泪,2009-01-16,2009,静茹 & 情歌 别再为他流泪,PART 1,0.0
1,411228,0,004K6Ne61a1VA8,会呼吸的痛,NaN,梁静茹,44,000GGDys0yA0Nk,33237,002oy2Mp3I8Rgo,272,1194537600,会呼吸的痛,会呼吸的痛,崇拜,2007-11-09,2007,崇拜,PART 1,0.0
2,411231,0,002x8dpU2QNXFP,给未来的自己,NaN,梁静茹,44,000GGDys0yA0Nk,33237,002oy2Mp3I8Rgo,250,1194537600,给未来的自己,给未来的自己,崇拜,2007-11-09,2007,崇拜,PART 1,0.0
3,4830164,0,000LDr7E13dXy1,勇气,《侠女闯天关》电视剧台湾版主题曲|《出水芙蓉》电视剧片尾曲,梁静茹,44,000GGDys0yA0Nk,96253,001DEgPu1004bl,239,965145600,勇气,勇气,勇气,2000-08-02,2000,勇气,PART 1,0.0
4,169763,0,003Sn9Rg4N3oNq,暖暖,《周末父母》电视剧片头曲,梁静茹,44,000GGDys0yA0Nk,14922,004R8LFN08EGAD,243,1160064000,暖暖,暖暖,亲亲,2006-10-06,2006,亲亲,PART 1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,4931465,0,003WPSKz2aIPa6,看海计划,NaN,梁静茹,44,000GGDys0yA0Nk,96292,003MSjCv4OkruJ,241,993657600,看海计划,看海计划,闪亮的星,2001-06-28,2001,闪亮的星,PART 10,9.0
96,462189,0,003uBcpH3LMMkU,不敢当,NaN,梁静茹,44,000GGDys0yA0Nk,37603,002E4IXe1ESUij,252,1232035200,不敢当,不敢当,静茹 & 情歌 别再为他流泪,2009-01-16,2009,静茹 & 情歌 别再为他流泪,PART 10,9.0
97,412406449,0,001ZVOtt2QMObS,麋鹿,NaN,梁静茹,44,000GGDys0yA0Nk,38203256,0045kqXc2yaLBr,205,1684339200,麋鹿,麋鹿,麋鹿,2023-05-18,2023,麋鹿,PART 10,9.0
98,412406455,0,001Er0w73uJvop,宁静海,NaN,梁静茹,44,000GGDys0yA0Nk,38203256,0045kqXc2yaLBr,272,1684339200,宁静海,宁静海,麋鹿,2023-05-18,2023,麋鹿,PART 10,9.0


In [91]:
df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

# 测试

In [ ]:
1260/500

In [ ]:
2318/2.52

In [ ]:
from datetime import datetime
datetime.now().month